# StarDist Cell Segmentation - Kaggle Training Notebook

**Attention-Enhanced U-Net with 3 Output Heads (Prob + Dist + Boundary)**

## Cấu hình tối ưu:
- **Parameters:** 126K (giảm 93.7% so với baseline 2M)
- **Architecture:** SE Attention + Attention Gates + Boundary Head
- **Loss:** BCE + MAE + Boundary Dice Loss

## Kaggle Setup:
1. GPU: Settings → Accelerator → GPU (T4/P100)
2. Data: Upload dataset hoặc add Kaggle dataset
3. Internet: Enable if needed for pip install

## 1. Setup & Installation

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

In [ ]:
# Install dependencies (nếu cần)
!pip install -q tifffile scikit-image scipy tqdm tensorboard

In [ ]:
pip install sympy==1.13.3

In [ ]:

import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

print("✓ Dependencies installed")

## 2. Data Loading

### Option A: Upload code files từ local
Nếu bạn upload project lên Kaggle dataset, uncomment dòng dưới:

In [ ]:
# Option A: Nếu đã upload code lên Kaggle dataset
# CODE_DIR = "/kaggle/input/stardist-pytorch-code"
# sys.path.insert(0, CODE_DIR)

# Option B: Paste code trực tiếp (cho demo)
# Sử dụng current directory
sys.path.insert(0, '.')

### Data Directory Setup

**Kaggle Dataset Format:**
```
/kaggle/input/your-dataset/
├── train/
│   ├── images/
│   │   ├── img_001.tif
│   │   └── ...
│   └── masks/
│       ├── img_001.tif
│       └── ...
└── test/ (optional)
```

In [ ]:
# Cấu hình paths
# Thay đổi path này theo dataset của bạn
DATA_ROOT = "/kaggle/input/datasets/minhduc0912/dsb2018/data/dsb2018/train"  # Thay bằng path của bạn

# Hoặc sử dụng DSB2018 public dataset
# DATA_ROOT = "/kaggle/input/data-science-bowl-2018/stage1_train"

# Working directory cho outputs
WORK_DIR = "/kaggle/working"
os.makedirs(WORK_DIR, exist_ok=True)

print(f"Data directory: {DATA_ROOT}")
print(f"Working directory: {WORK_DIR}")

## 3. Model Definitions

Copy paste các modules chính từ project

In [ ]:
# ============================================================================
# MODELS.PY - Attention-Enhanced U-Net
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

# Squeeze-and-Excitation Block (Channel Attention)
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        b, c, _, _ = x.size()
        z = self.squeeze(x).view(b, c)
        s = self.excitation(z).view(b, c, 1, 1)
        return x * s.expand_as(x)

# Attention Gate (Spatial Attention for Skip Connections)
class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)
        
    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

# CBAM: Convolutional Block Attention Module
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        out = self.conv(x_cat)
        return self.sigmoid(out)

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = ChannelAttention(channels, reduction)
        self.spatial_att = SpatialAttention(kernel_size)
        
    def forward(self, x):
        x = x * self.channel_att(x)
        x = x * self.spatial_att(x)
        return x

# Double Conv with Attention
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, use_attention='se'):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
        
        if use_attention == 'se':
            self.attention = SEBlock(out_ch)
        elif use_attention == 'cbam':
            self.attention = CBAM(out_ch)
        else:
            self.attention = nn.Identity()

    def forward(self, x):
        x = self.conv(x)
        x = self.attention(x)
        return x

# StarDist2D Model
class StarDist2D(nn.Module):
    def __init__(
        self,
        n_channels_in=1,
        n_rays=32,
        grid=(1, 1),
        unet_n_filter_base=16,
        unet_n_depth=2,
        net_conv_after_unet=32,
        use_attention='se',
        use_attention_gate=True,
        use_boundary_head=True,
    ):
        super().__init__()
        self.n_rays = n_rays
        self.use_attention_gate = use_attention_gate
        self.use_boundary_head = use_boundary_head

        # Encoder
        self.down_blocks = nn.ModuleList()
        self.pools = nn.ModuleList()
        
        in_ch = n_channels_in
        out_ch = unet_n_filter_base
        encoder_channels = []

        for _ in range(unet_n_depth):
            self.down_blocks.append(DoubleConv(in_ch, out_ch, use_attention=use_attention))
            self.pools.append(nn.MaxPool2d(2))
            encoder_channels.append(out_ch)
            in_ch = out_ch
            out_ch *= 2

        # Bottleneck
        self.bottleneck = DoubleConv(in_ch, out_ch, use_attention=use_attention)

        # Attention Gates
        self.attention_gates = nn.ModuleList()
        if use_attention_gate:
            temp_ch = out_ch
            for i in range(unet_n_depth):
                F_g = temp_ch // 2
                F_l = encoder_channels[-(i+1)]
                F_int = F_l // 2
                self.attention_gates.append(AttentionGate(F_g, F_l, F_int))
                temp_ch = temp_ch // 2
        
        # Decoder
        self.up_transpose = nn.ModuleList()
        self.up_blocks = nn.ModuleList()

        for _ in range(unet_n_depth):
            self.up_transpose.append(nn.ConvTranspose2d(out_ch, out_ch // 2, 2, stride=2))
            self.up_blocks.append(DoubleConv(out_ch, out_ch // 2, use_attention=use_attention))
            out_ch //= 2

        # Extra Conv
        if net_conv_after_unet > 0:
            self.features = nn.Sequential(
                nn.Conv2d(out_ch, net_conv_after_unet, 3, padding=1),
                nn.ReLU(inplace=True)
            )
            final_ch = net_conv_after_unet
        else:
            self.features = nn.Identity()
            final_ch = out_ch

        # Output Heads
        self.prob_head = nn.Sequential(nn.Conv2d(final_ch, 1, 1), nn.Sigmoid())
        self.dist_head = nn.Conv2d(final_ch, n_rays, 1)
        
        if use_boundary_head:
            self.boundary_head = nn.Sequential(nn.Conv2d(final_ch, 1, 1), nn.Sigmoid())

    def forward(self, x):
        # Encoder
        skips = []
        for down, pool in zip(self.down_blocks, self.pools):
            x = down(x)
            skips.append(x)
            x = pool(x)

        # Bottleneck
        x = self.bottleneck(x)

        # Decoder
        for i, (up_trans, up_block, skip) in enumerate(zip(
            self.up_transpose, self.up_blocks, reversed(skips)
        )):
            x = up_trans(x)
            
            if self.use_attention_gate:
                skip = self.attention_gates[i](g=x, x=skip)

            diffY = skip.size(2) - x.size(2)
            diffX = skip.size(3) - x.size(3)
            x = F.pad(x, [diffX//2, diffX - diffX//2, diffY//2, diffY - diffY//2])

            x = torch.cat([skip, x], dim=1)
            x = up_block(x)

        feat = self.features(x)

        outputs = {
            'prob': self.prob_head(feat),
            'dist': self.dist_head(feat),
        }
        
        if self.use_boundary_head:
            outputs['boundary'] = self.boundary_head(feat)

        return outputs

print("✓ Model architecture loaded")

## 4. Loss Functions

In [ ]:
# ============================================================================
# LOSS.PY - Combined Loss Functions
# ============================================================================

def masked_bce_loss():
    def bce(y_true, y_pred):
        valid_mask = (y_true >= 0).float()
        y_true = torch.clamp(y_true, 0.0, 1.0)
        y_pred = torch.clamp(y_pred, 1e-7, 1.0 - 1e-7)
        bce_loss = F.binary_cross_entropy(y_pred, y_true, reduction='none')
        masked_loss = torch.sum(bce_loss * valid_mask) / (torch.sum(valid_mask) + 1e-8)
        return masked_loss
    return bce

def generic_masked_loss(mask, loss_fn, weights=1.0, norm_by_mask=True, reg_weight=0.0):
    def _loss(y_true, y_pred):
        m = mask.float()
        w = torch.as_tensor(weights, device=y_true.device, dtype=torch.float32)
        per_pixel_loss = loss_fn(y_true, y_pred)
        actual_loss = torch.mean(m * w * per_pixel_loss, dim=[1,2,3])
        norm_mask = (torch.mean(m, dim=[1,2,3]) + 1e-8) if norm_by_mask else 1.0
        normalized_loss = actual_loss / norm_mask
        
        if reg_weight > 0:
            reg_loss = torch.mean((1 - m) * torch.abs(y_pred), dim=[1,2,3])
            total_loss_val = normalized_loss + reg_weight * reg_loss
        else:
            total_loss_val = normalized_loss
        
        return total_loss_val.mean()
    return _loss

def masked_mae_loss(mask, reg_weight=1e-4, norm_by_mask=True):
    def mae_loss(y_true, y_pred):
        return torch.abs(y_true - y_pred)
    return generic_masked_loss(mask=mask, loss_fn=mae_loss, weights=1.0, 
                               norm_by_mask=norm_by_mask, reg_weight=reg_weight)

def boundary_dice_loss(y_true, y_pred, smooth=1e-5):
    """Boundary Dice Loss for class imbalance"""
    y_true_f = y_true.view(y_true.size(0), -1)
    y_pred_f = y_pred.view(y_pred.size(0), -1)
    
    intersection = torch.sum(y_pred_f * y_true_f, dim=1)
    sum_pred = torch.sum(y_pred_f, dim=1)
    sum_true = torch.sum(y_true_f, dim=1)
    
    dice = (2.0 * intersection + smooth) / (sum_pred + sum_true + smooth)
    loss = 1.0 - dice
    return loss.mean()

def total_loss_with_boundary(prob_pred, dist_pred, boundary_pred, 
                             prob_gt, dist_mask_gt, boundary_gt,
                             loss_weights=(1.0, 0.2, 0.5)):
    """Combined Loss: BCE + MAE + Boundary Dice"""
    # Probability Loss
    prob_loss_fn = masked_bce_loss()
    p_loss = prob_loss_fn(prob_gt, prob_pred)
    
    # Distance Loss
    mask = dist_mask_gt[:, -1:]
    dist_gt = dist_mask_gt[:, :-1]
    dist_loss_fn = masked_mae_loss(mask=mask)
    d_loss = dist_loss_fn(dist_gt, dist_pred)
    
    # Boundary Dice Loss
    b_loss = boundary_dice_loss(boundary_gt, boundary_pred)
    
    total = (loss_weights[0] * p_loss + 
             loss_weights[1] * d_loss + 
             loss_weights[2] * b_loss)
    
    return total, (p_loss.item(), d_loss.item(), b_loss.item())

print("✓ Loss functions loaded")

## 5. Data Loading & Preprocessing

Simplified dataset cho Kaggle (không cần phức tạp)

In [ ]:
# ============================================================================
# SIMPLE DATASET - For Kaggle
# ============================================================================

import glob
import tifffile
from torch.utils.data import Dataset, DataLoader, random_split
from scipy.ndimage import distance_transform_edt, maximum_filter
from skimage import morphology

def ray_angles(n_rays=32):
    return np.linspace(0, 2 * np.pi, n_rays, endpoint=False)

def star_dist(mask, n_rays=32):
    """Compute radial distances from object centers"""
    from skimage.measure import regionprops
    
    dist = np.zeros((n_rays, *mask.shape), dtype=np.float32)
    
    for region in regionprops(mask):
        cy, cx = region.centroid
        coords = region.coords  # (N, 2) array of y, x
        
        angles = ray_angles(n_rays)
        for i, angle in enumerate(angles):
            dy = np.sin(angle)
            dx = np.cos(angle)
            
            max_dist = 0
            for y, x in coords:
                proj = (y - cy) * dy + (x - cx) * dx
                if proj > max_dist:
                    max_dist = proj
            
            if max_dist > 0:
                dist[i, int(cy), int(cx)] = max_dist
    
    return dist

def edt_prob(mask):
    """Compute probability map from EDT"""
    from skimage.segmentation import find_boundaries
    
    prob = np.zeros_like(mask, dtype=np.float32)
    
    for region_id in np.unique(mask):
        if region_id == 0:
            continue
        region_mask = (mask == region_id)
        edt = distance_transform_edt(region_mask)
        if edt.max() > 0:
            prob = np.maximum(prob, edt / edt.max())
    
    return prob

def compute_boundary(mask):
    """Compute boundary from instance mask"""
    from skimage.segmentation import find_boundaries
    boundary = find_boundaries(mask, mode='inner').astype(np.float32)
    return boundary

class SimpleStarDistDataset(Dataset):
    """Simplified dataset for Kaggle"""
    def __init__(self, image_paths, mask_paths, patch_size=(256, 256), n_rays=32):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.patch_size = patch_size
        self.n_rays = n_rays
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        # Load image and mask
        img = tifffile.imread(self.image_paths[idx]).astype(np.float32)
        mask = tifffile.imread(self.mask_paths[idx]).astype(np.int32)
        
        # Normalize image
        if img.max() > 0:
            img = img / img.max()
        
        # Random crop
        h, w = img.shape
        ph, pw = self.patch_size
        
        if h > ph and w > pw:
            y = np.random.randint(0, h - ph)
            x = np.random.randint(0, w - pw)
            img = img[y:y+ph, x:x+pw]
            mask = mask[y:y+ph, x:x+pw]
        else:
            # Pad if too small
            img = np.pad(img, [(0, max(0, ph-h)), (0, max(0, pw-w))], mode='constant')
            mask = np.pad(mask, [(0, max(0, ph-h)), (0, max(0, pw-w))], mode='constant')
            img = img[:ph, :pw]
            mask = mask[:ph, :pw]
        
        # Compute targets
        prob = edt_prob(mask)
        dist = star_dist(mask, self.n_rays)  # (n_rays, H, W)
        boundary = compute_boundary(mask)
        
        # Create mask for distance (where cells exist)
        dist_mask = (mask > 0).astype(np.float32)
        
        # Convert to tensors (FORCE FLOAT32 to avoid dtype mismatch)
        img_t = torch.from_numpy(img).float().unsqueeze(0)  # (1, H, W)
        prob_t = torch.from_numpy(prob).float().unsqueeze(0)  # (1, H, W)
        dist_t = torch.from_numpy(dist).float()  # (n_rays, H, W)
        dist_mask_t = torch.from_numpy(dist_mask).float().unsqueeze(0)  # (1, H, W)
        dist_with_mask = torch.cat([dist_t, dist_mask_t], dim=0)  # (n_rays+1, H, W)
        boundary_t = torch.from_numpy(boundary).float().unsqueeze(0)  # (1, H, W)
        
        return img_t, prob_t, dist_with_mask, boundary_t

def create_kaggle_dataloaders(data_root, batch_size=8, val_split=0.2, num_workers=2):
    """Create train/val dataloaders for Kaggle"""
    
    # Find all images and masks
    # Adjust pattern based on your data structure
    image_pattern = os.path.join(data_root, "*/images/*.tif")  # DSB2018 format
    
    image_paths = sorted(glob.glob(image_pattern))
    
    if len(image_paths) == 0:
        # Try alternative pattern
        image_pattern = os.path.join(data_root, "images/*.tif")
        image_paths = sorted(glob.glob(image_pattern))
    
    # Find corresponding masks
    mask_paths = []
    for img_path in image_paths:
        # Try to find mask
        mask_path = img_path.replace("/images/", "/masks/")
        if not os.path.exists(mask_path):
            # Alternative: masks/ directory at same level
            base_dir = os.path.dirname(os.path.dirname(img_path))
            img_name = os.path.basename(img_path)
            mask_path = os.path.join(base_dir, "masks", img_name)
        mask_paths.append(mask_path)
    
    # Verify paths exist
    valid_pairs = [(img, mask) for img, mask in zip(image_paths, mask_paths) 
                   if os.path.exists(img) and os.path.exists(mask)]
    
    if len(valid_pairs) == 0:
        raise ValueError(f"No valid image-mask pairs found in {data_root}")
    
    image_paths, mask_paths = zip(*valid_pairs)
    
    print(f"Found {len(image_paths)} image-mask pairs")
    
    # Create dataset
    dataset = SimpleStarDistDataset(image_paths, mask_paths)
    
    # Split train/val
    val_size = int(len(dataset) * val_split)
    train_size = len(dataset) - val_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                             num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, 
                           num_workers=num_workers, pin_memory=True)
    
    return train_loader, val_loader

print("✓ Dataset utilities loaded")

## 6. Training Configuration

In [ ]:
# ============================================================================
# TRAINING CONFIGURATION
# ============================================================================

class Config:
    # Training parameters
    epochs = 100
    batch_size = 8  # Giảm batch size cho 2M model
    learning_rate = 0.0003
    patch_size = (256, 256)
    n_rays = 32
    
    # Loss weights (λ1_prob, λ2_dist, λ3_boundary)
    loss_weights = (1.0, 0.2, 0.5)  # Balanced
    
    # Model architecture (FULL POWER - 2M params cho Kaggle GPU)
    unet_n_depth = 3           # Full depth
    unet_n_filter_base = 32    # Full base filters  
    net_conv_after_unet = 128  # Full conv filters
    
    # Attention mechanisms
    use_attention = 'se'       # 'se', 'cbam', or None
    use_attention_gate = True
    use_boundary_head = True
    
    # Paths
    data_root = DATA_ROOT
    save_dir = os.path.join(WORK_DIR, "checkpoints")
    log_dir = os.path.join(WORK_DIR, "logs")
    
    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Early stopping
    early_stop_patience = 20

config = Config()
os.makedirs(config.save_dir, exist_ok=True)
os.makedirs(config.log_dir, exist_ok=True)

print("\n" + "="*60)
print("CONFIGURATION")
print("="*60)
print(f"Device: {config.device}")
print(f"Batch size: {config.batch_size}")
print(f"Epochs: {config.epochs}")
print(f"Model: depth={config.unet_n_depth}, base={config.unet_n_filter_base}, conv={config.net_conv_after_unet}")
print(f"Attention: {config.use_attention}, AG={config.use_attention_gate}, Boundary={config.use_boundary_head}")
print(f"Loss weights: {config.loss_weights}")
print("="*60 + "\n")

## 7. Load Data

In [ ]:
print("Loading data...")

try:
    train_loader, val_loader = create_kaggle_dataloaders(
        data_root=config.data_root,
        batch_size=config.batch_size,
        val_split=0.2,
        num_workers=2
    )
    
    print(f"✓ Train samples: {len(train_loader.dataset)}")
    print(f"✓ Val samples: {len(val_loader.dataset)}")
    print(f"✓ Train batches: {len(train_loader)}")
    print(f"✓ Val batches: {len(val_loader)}")
    
    # Test một batch
    img, prob, dist_mask, boundary = next(iter(train_loader))
    print(f"\nSample batch shapes:")
    print(f"  Image: {img.shape}")
    print(f"  Prob: {prob.shape}")
    print(f"  Dist+Mask: {dist_mask.shape}")
    print(f"  Boundary: {boundary.shape}")
    
except Exception as e:
    print(f"❌ Error loading data: {e}")
    print(f"\nPlease check:")
    print(f"1. DATA_ROOT is correct: {config.data_root}")
    print(f"2. Data structure matches expected format")
    raise

## 8. Initialize Model

In [ ]:
print("Initializing model...")

model = StarDist2D(
    n_channels_in=1,
    n_rays=config.n_rays,
    unet_n_depth=config.unet_n_depth,
    unet_n_filter_base=config.unet_n_filter_base,
    net_conv_after_unet=config.net_conv_after_unet,
    use_attention=config.use_attention,
    use_attention_gate=config.use_attention_gate,
    use_boundary_head=config.use_boundary_head,
).to(config.device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n{'='*60}")
print(f"MODEL SUMMARY")
print(f"{'='*60}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Expected: ~2M parameters ✓" if 1900000 < total_params < 2100000 else f"⚠️ Unexpected parameter count")
print(f"{'='*60}\n")

# Optimizer & Scheduler
optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=config.epochs, eta_min=1e-6
)

print("✓ Optimizer: Adam")
print("✓ Scheduler: CosineAnnealingLR")


## 9. Training Loop

In [ ]:

def inference_with_tta(model, x, n_rays=32):
    """StarDist Test-Time Augmentation (TTA) with proper ray angle mapping"""
    b, c, h, w = x.shape
    
    # 1. Original
    out1 = model(x)
    
    # 2. H-flip
    x_h = torch.flip(x, dims=[3])
    out2 = model(x_h)
    p2 = torch.flip(out2['prob'], dims=[3])
    d2 = torch.flip(out2['dist'], dims=[3])
    h_idx = [(n_rays // 2 - i) % n_rays for i in range(n_rays)]
    d2 = d2[:, h_idx, :, :]
    
    # 3. V-flip
    x_v = torch.flip(x, dims=[2])
    out3 = model(x_v)
    p3 = torch.flip(out3['prob'], dims=[2])
    d3 = torch.flip(out3['dist'], dims=[2])
    v_idx = [(n_rays - i) % n_rays for i in range(n_rays)]
    d3 = d3[:, v_idx, :, :]
    
    # 4. H+V flip (180 deg rotation)
    x_hv = torch.flip(x, dims=[2, 3])
    out4 = model(x_hv)
    p4 = torch.flip(out4['prob'], dims=[2, 3])
    d4 = torch.flip(out4['dist'], dims=[2, 3])
    hv_idx = [(n_rays - ((n_rays // 2 - i) % n_rays)) % n_rays for i in range(n_rays)]
    d4 = d4[:, hv_idx, :, :]
    
    res = {
        'prob': (out1['prob'] + p2 + p3 + p4) / 4.0,
        'dist': (out1['dist'] + d2 + d3 + d4) / 4.0
    }
    if 'boundary' in out1:
        b2 = torch.flip(out2['boundary'], dims=[3])
        b3 = torch.flip(out3['boundary'], dims=[2])
        b4 = torch.flip(out4['boundary'], dims=[2, 3])
        res['boundary'] = (out1['boundary'] + b2 + b3 + b4) / 4.0
    return res
# ============================================================================
# TRAINING LOOP
# ============================================================================

def train_one_epoch(model, train_loader, optimizer, device, loss_weights):
    model.train()
    total_loss = 0
    components = np.zeros(3)
    
    pbar = tqdm(train_loader, desc="Training", leave=False)
    for images, prob_gt, dist_mask_gt, boundary_gt in pbar:
        images = images.to(device)
        prob_gt = prob_gt.to(device)
        dist_mask_gt = dist_mask_gt.to(device)
        boundary_gt = boundary_gt.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        prob_pred = outputs['prob']
        dist_pred = outputs['dist']
        boundary_pred = outputs['boundary']
        
        loss, comps = total_loss_with_boundary(
            prob_pred, dist_pred, boundary_pred,
            prob_gt, dist_mask_gt, boundary_gt,
            loss_weights=loss_weights
        )
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        components += np.array(comps)
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(train_loader), components / len(train_loader)

def validate(model, val_loader, device, loss_weights):
    model.eval()
    total_loss = 0
    components = np.zeros(3)
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc="Validation", leave=False)
        for images, prob_gt, dist_mask_gt, boundary_gt in pbar:
            images = images.to(device)
            prob_gt = prob_gt.to(device)
            dist_mask_gt = dist_mask_gt.to(device)
            boundary_gt = boundary_gt.to(device)
            
            outputs = inference_with_tta(model, images, config.n_rays)
            prob_pred = outputs['prob']
            dist_pred = outputs['dist']
            boundary_pred = outputs['boundary']
            
            loss, comps = total_loss_with_boundary(
                prob_pred, dist_pred, boundary_pred,
                prob_gt, dist_mask_gt, boundary_gt,
                loss_weights=loss_weights
            )
            
            total_loss += loss.item()
            components += np.array(comps)
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(val_loader), components / len(val_loader)

print("✓ Training functions defined")

In [ ]:
# ============================================================================
# START TRAINING
# ============================================================================

print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60)

best_val_loss = float('inf')
patience_counter = 0
history = {
    'train_loss': [],
    'val_loss': [],
    'train_prob': [],
    'train_dist': [],
    'train_boundary': [],
    'val_prob': [],
    'val_dist': [],
    'val_boundary': [],
}

for epoch in range(1, config.epochs + 1):
    start_time = time.time()
    
    # Train
    train_loss, train_comps = train_one_epoch(
        model, train_loader, optimizer, config.device, config.loss_weights
    )
    
    # Validate
    val_loss, val_comps = validate(
        model, val_loader, config.device, config.loss_weights
    )
    
    # Scheduler step
    scheduler.step()
    
    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_prob'].append(train_comps[0])
    history['train_dist'].append(train_comps[1])
    history['train_boundary'].append(train_comps[2])
    history['val_prob'].append(val_comps[0])
    history['val_dist'].append(val_comps[1])
    history['val_boundary'].append(val_comps[2])
    
    # Print progress
    epoch_time = time.time() - start_time
    print(f"Epoch {epoch}/{config.epochs} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Prob: {train_comps[0]:.3f} | Dist: {train_comps[1]:.3f} | Bound: {train_comps[2]:.3f} | "
          f"Time: {epoch_time:.1f}s | LR: {optimizer.param_groups[0]['lr']:.2e}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'config': config.__dict__,
        }, os.path.join(config.save_dir, 'best_model.pth'))
        print(f"  → Saved best model (val_loss = {val_loss:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= config.early_stop_patience:
            print(f"\nEarly stopping at epoch {epoch}")
            break
    
    # Save checkpoint every 10 epochs
    if epoch % 10 == 0:
        torch.save(model.state_dict(), 
                  os.path.join(config.save_dir, f'model_epoch_{epoch}.pth'))

print("\n" + "="*60)
print("TRAINING COMPLETED")
print("="*60)
print(f"Best validation loss: {best_val_loss:.4f}")

## 10. Plot Training History

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Total Loss
axes[0, 0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0, 0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0, 0].set_title('Total Loss', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Probability Loss
axes[0, 1].plot(history['train_prob'], label='Train Prob', linewidth=2)
axes[0, 1].plot(history['val_prob'], label='Val Prob', linewidth=2)
axes[0, 1].set_title('Probability Loss (BCE)', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Distance Loss
axes[1, 0].plot(history['train_dist'], label='Train Dist', linewidth=2)
axes[1, 0].plot(history['val_dist'], label='Val Dist', linewidth=2)
axes[1, 0].set_title('Distance Loss (MAE)', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Boundary Loss
axes[1, 1].plot(history['train_boundary'], label='Train Boundary', linewidth=2)
axes[1, 1].plot(history['val_boundary'], label='Val Boundary', linewidth=2)
axes[1, 1].set_title('Boundary Loss (Dice)', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(config.save_dir, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Training history saved to {config.save_dir}/training_history.png")

## 11. Evaluation on Multiple Thresholds

Đánh giá model với nhiều ngưỡng khác nhau

In [ ]:
# ============================================================================
# EVALUATION FUNCTIONS
# ============================================================================

from scipy.ndimage import label as scipy_label
from skimage.measure import regionprops

def non_maximum_suppression(prob, dist, prob_thresh=0.5, nms_thresh=0.4):
    """
    Simple NMS for cell detection
    
    Args:
        prob: (H, W) probability map
        dist: (n_rays, H, W) distance map
        prob_thresh: probability threshold
        nms_thresh: NMS IoU threshold
    
    Returns:
        mask: (H, W) instance segmentation mask
    """
    # Find local maxima above threshold
    prob_binary = prob > prob_thresh
    
    # Label connected components
    labeled, num_features = scipy_label(prob_binary)
    
    # Create instance mask (placeholder - full implementation needs polygon drawing)
    # For evaluation, we use the probability peaks as approximation
    return labeled

def compute_iou(mask1, mask2):
    """Compute IoU between two binary masks"""
    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    return intersection / (union + 1e-8)

def evaluate_detection(gt_mask, pred_mask, iou_thresh=0.5):
    """
    Evaluate detection performance
    
    Returns:
        precision, recall, f1
    """
    from skimage.measure import regionprops
    
    gt_regions = regionprops(gt_mask)
    pred_regions = regionprops(pred_mask)
    
    n_gt = len(gt_regions)
    n_pred = len(pred_regions)
    
    if n_gt == 0 and n_pred == 0:
        return 1.0, 1.0, 1.0
    if n_gt == 0:
        return 0.0, 1.0, 0.0
    if n_pred == 0:
        return 1.0, 0.0, 0.0
    
    # Match predictions to ground truth
    matched_gt = set()
    matched_pred = set()
    
    for i, pred_region in enumerate(pred_regions):
        pred_mask_i = (pred_mask == pred_region.label)
        
        best_iou = 0
        best_j = -1
        
        for j, gt_region in enumerate(gt_regions):
            if j in matched_gt:
                continue
            
            gt_mask_j = (gt_mask == gt_region.label)
            iou = compute_iou(pred_mask_i, gt_mask_j)
            
            if iou > best_iou:
                best_iou = iou
                best_j = j
        
        if best_iou >= iou_thresh:
            matched_gt.add(best_j)
            matched_pred.add(i)
    
    tp = len(matched_pred)
    fp = n_pred - tp
    fn = n_gt - len(matched_gt)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
    
    return precision, recall, f1, accuracy

print("✓ Evaluation functions loaded")


def compute_dsb2018_ap(gt_mask, pred_mask, thresholds=[0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]):
    from skimage.measure import regionprops
    import numpy as np
    
    gt_regions = regionprops(gt_mask)
    pred_regions = regionprops(pred_mask)
    n_gt = len(gt_regions)
    n_pred = len(pred_regions)
    
    if n_gt == 0 and n_pred == 0: return 1.0
    if n_gt == 0 or n_pred == 0: return 0.0
        
    iou_matrix = np.zeros((n_pred, n_gt))
    for i, pred_region in enumerate(pred_regions):
        pred_mask_i = (pred_mask == pred_region.label)
        p_min_row, p_min_col, p_max_row, p_max_col = pred_region.bbox
        for j, gt_region in enumerate(gt_regions):
            g_min_row, g_min_col, g_max_row, g_max_col = gt_region.bbox
            if (p_min_row >= g_max_row or p_max_row <= g_min_row or 
                p_min_col >= g_max_col or p_max_col <= g_min_col):
                continue
                
            gt_mask_j = (gt_mask == gt_region.label)
            overlap = np.logical_and(pred_mask_i, gt_mask_j)
            if overlap.any():
                union = np.logical_or(pred_mask_i, gt_mask_j).sum()
                iou_matrix[i, j] = overlap.sum() / union

    aps = []
    for thresh in thresholds:
        tp = 0
        iou_mat = iou_matrix.copy()
        
        for i in range(n_pred):
            best_iou = 0
            best_j = -1
            for j in range(n_gt):
                if iou_mat[i, j] > best_iou:
                    best_iou = iou_mat[i, j]
                    best_j = j
            if best_iou >= thresh:
                tp += 1
                iou_mat[:, best_j] = 0 # Block this GT point since it's already matched
                
        fp = n_pred - tp
        fn = n_gt - tp
        ap = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
        aps.append(ap)
        
    return np.mean(aps)

print("✓ AP evaluation function loaded")


In [ ]:
# ============================================================================
# EVALUATE ON MULTIPLE THRESHOLDS
# ============================================================================

print("\n" + "="*60)
print("EVALUATION ON MULTIPLE THRESHOLDS")
print("="*60)

# Load best model
checkpoint = torch.load(os.path.join(config.save_dir, 'best_model.pth'))
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"✓ Loaded best model from epoch {checkpoint['epoch']}\n")

# Test thresholds
prob_thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
nms_thresholds = [0.3, 0.4, 0.5]

results = []

with torch.no_grad():
    for prob_thresh in prob_thresholds:
        for nms_thresh in nms_thresholds:
            precisions = []
            accuracies = []
            recalls = []
            f1_scores = []
            ap_scores = []
            
            pbar = tqdm(val_loader, desc=f"Prob={prob_thresh}, NMS={nms_thresh}", leave=False)
            
            for images, prob_gt, dist_mask_gt, boundary_gt in pbar:
                images = images.to(config.device)
                
                outputs = inference_with_tta(model, images, config.n_rays)
                prob_pred = outputs['prob'].cpu().numpy()
                dist_pred = outputs['dist'].cpu().numpy()
                
                # Evaluate each image in batch
                for i in range(len(images)):
                    # Ground truth mask (from dist_mask_gt)
                    gt_mask = (dist_mask_gt[i, -1].cpu().numpy() > 0).astype(np.int32)
                    gt_labeled, _ = scipy_label(gt_mask)
                    
                    # Predicted mask
                    pred_prob = prob_pred[i, 0]
                    pred_dist = dist_pred[i]
                    pred_mask = non_maximum_suppression(
                        pred_prob, pred_dist, 
                        prob_thresh=prob_thresh, 
                        nms_thresh=nms_thresh
                    )
                    
                    # Compute metrics
                    prec, rec, f1, acc = evaluate_detection(gt_labeled, pred_mask, iou_thresh=0.5)
                    ap_score = compute_dsb2018_ap(gt_labeled, pred_mask)\n
                    precisions.append(prec)
                    recalls.append(rec)
                    f1_scores.append(f1)
                    accuracies.append(acc)
                    ap_scores.append(ap_score)\n
            
            # Average metrics
            avg_prec = np.mean(precisions)
            avg_rec = np.mean(recalls)
            avg_f1 = np.mean(f1_scores)
            avg_acc = np.mean(accuracies)
            avg_ap = np.mean(ap_scores)\n
            
            results.append({
                'prob_thresh': prob_thresh,
                'nms_thresh': nms_thresh,
                'precision': avg_prec,
                'recall': avg_rec,
                'f1': avg_f1,\n
                'accuracy': avg_acc,
                'ap': avg_ap\n
            })
            
            print(f"Prob={prob_thresh:.1f}, NMS={nms_thresh:.1f} → "
                  f"Precision={avg_prec:.3f}, Recall={avg_rec:.3f}, F1={avg_f1:.3f}, AP={avg_ap:.3f}, Acc={avg_acc:.3f}")

# Convert to DataFrame for easy analysis
import pandas as pd
results_df = pd.DataFrame(results)

print("\n" + "="*60)
print("BEST CONFIGURATIONS")
print("="*60)

# Find best by F1
best_f1 = results_df.loc[results_df['f1'].idxmax()]
print(f"\nBest F1 Score: {best_f1['f1']:.3f}")
print(f"  - Prob threshold: {best_f1['prob_thresh']}")
print(f"  - NMS threshold: {best_f1['nms_thresh']}")
print(f"  - Precision: {best_f1['precision']:.3f}")
print(f"  - Recall: {best_f1['recall']:.3f}")
print(f"  - Best Accuracy: {best_f1['accuracy']:.3f}")

# Save results
results_df.to_csv(os.path.join(config.save_dir, 'threshold_evaluation.csv'), index=False)
print(f"\n✓ Results saved to {config.save_dir}/threshold_evaluation.csv")

## 12. Visualize Results

In [ ]:
# Plot threshold analysis
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for nms_t in nms_thresholds:
    subset = results_df[results_df['nms_thresh'] == nms_t]
    axes[0].plot(subset['prob_thresh'], subset['precision'], marker='o', label=f'NMS={nms_t}')
    axes[1].plot(subset['prob_thresh'], subset['recall'], marker='o', label=f'NMS={nms_t}')
    axes[2].plot(subset['prob_thresh'], subset['f1'], marker='o', label=f'NMS={nms_t}')

axes[0].set_title('Precision vs Prob Threshold', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Probability Threshold')
axes[0].set_ylabel('Precision')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Recall vs Prob Threshold', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Probability Threshold')
axes[1].set_ylabel('Recall')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].set_title('F1 Score vs Prob Threshold', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Probability Threshold')
axes[2].set_ylabel('F1 Score')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(config.save_dir, 'threshold_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Threshold analysis saved to {config.save_dir}/threshold_analysis.png")

In [ ]:
# Visualize predictions on sample images
fig, axes = plt.subplots(3, 5, figsize=(20, 12))

model.eval()
with torch.no_grad():
    # Get one batch
    images, prob_gt, dist_mask_gt, boundary_gt = next(iter(val_loader))
    images_gpu = images.to(config.device)
    
    outputs = model(images_gpu)
    prob_pred = outputs['prob'].cpu().numpy()
    dist_pred = outputs['dist'].cpu().numpy()
    boundary_pred = outputs['boundary'].cpu().numpy()
    
    # Show first 5 samples
    for i in range(min(5, len(images))):
        # Original image
        axes[0, i].imshow(images[i, 0], cmap='gray')
        axes[0, i].set_title(f'Sample {i+1}', fontweight='bold')
        axes[0, i].axis('off')
        
        # Probability prediction
        axes[1, i].imshow(prob_pred[i, 0], cmap='hot')
        axes[1, i].set_title('Prob Pred', fontweight='bold')
        axes[1, i].axis('off')
        
        # Boundary prediction
        axes[2, i].imshow(boundary_pred[i, 0], cmap='hot')
        axes[2, i].set_title('Boundary Pred', fontweight='bold')
        axes[2, i].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(config.save_dir, 'predictions_sample.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Sample predictions saved to {config.save_dir}/predictions_sample.png")

## 13. Save Final Results & Summary

In [ ]:
# Create summary report
summary = f"""
{'='*70}
STARDIST CELL SEGMENTATION - TRAINING SUMMARY
{'='*70}

MODEL CONFIGURATION:
  - Architecture: Attention-Enhanced U-Net
  - Depth: {config.unet_n_depth}
  - Base filters: {config.unet_n_filter_base}
  - Conv after UNet: {config.net_conv_after_unet}
  - Total parameters: {total_params:,}
  - Attention: {config.use_attention}
  - Attention Gates: {config.use_attention_gate}
  - Boundary Head: {config.use_boundary_head}

TRAINING CONFIGURATION:
  - Epochs trained: {len(history['train_loss'])}
  - Batch size: {config.batch_size}
  - Learning rate: {config.learning_rate}
  - Loss weights: {config.loss_weights}
  - Train samples: {len(train_loader.dataset)}
  - Val samples: {len(val_loader.dataset)}

FINAL RESULTS:
  - Best validation loss: {best_val_loss:.4f}
  - Best F1 score: {best_f1['f1']:.3f}
  - Best AP score: {best_f1['ap']:.3f}\n
  - Best precision: {best_f1['precision']:.3f}
  - Best recall: {best_f1['recall']:.3f}
  - Optimal prob threshold: {best_f1['prob_thresh']}
  - Optimal NMS threshold: {best_f1['nms_thresh']}

OUTPUT FILES:
  - Best model: {config.save_dir}/best_model.pth
  - Training history: {config.save_dir}/training_history.png
  - Threshold evaluation: {config.save_dir}/threshold_evaluation.csv
  - Threshold analysis: {config.save_dir}/threshold_analysis.png
  - Sample predictions: {config.save_dir}/predictions_sample.png

{'='*70}
"""

print(summary)

# Save summary to file
with open(os.path.join(config.save_dir, 'training_summary.txt'), 'w') as f:
    f.write(summary)

print(f"\n✓ Summary saved to {config.save_dir}/training_summary.txt")
print(f"\n✅ ALL DONE! Check {config.save_dir}/ for all outputs.")

## 14. Test Set Evaluation

Đánh giá model trên test set với optimal thresholds

In [ ]:
# ============================================================================
# TEST SET EVALUATION
# ============================================================================

# Cấu hình test data path
TEST_DATA_ROOT = DATA_ROOT.replace("/train", "/test")  # Adjust based on your structure

# Hoặc set manual:
# TEST_DATA_ROOT = "/kaggle/input/your-dataset/test"

print("\n" + "="*60)
print("TEST SET EVALUATION")
print("="*60)
print(f"Test data path: {TEST_DATA_ROOT}")

# Check if test data exists
if os.path.exists(TEST_DATA_ROOT):
    print("✓ Test data found")
    
    # Create test dataloader
    try:
        # Find test images and masks
        test_image_pattern = os.path.join(TEST_DATA_ROOT, "*/images/*.tif")
        test_image_paths = sorted(glob.glob(test_image_pattern))
        
        if len(test_image_paths) == 0:
            test_image_pattern = os.path.join(TEST_DATA_ROOT, "images/*.tif")
            test_image_paths = sorted(glob.glob(test_image_pattern))
        
        test_mask_paths = []
        for img_path in test_image_paths:
            mask_path = img_path.replace("/images/", "/masks/")
            if not os.path.exists(mask_path):
                base_dir = os.path.dirname(os.path.dirname(img_path))
                img_name = os.path.basename(img_path)
                mask_path = os.path.join(base_dir, "masks", img_name)
            test_mask_paths.append(mask_path)
        
        # Verify
        valid_test_pairs = [(img, mask) for img, mask in zip(test_image_paths, test_mask_paths)
                           if os.path.exists(img) and os.path.exists(mask)]
        
        if len(valid_test_pairs) == 0:
            raise ValueError("No valid test image-mask pairs found")
        
        test_image_paths, test_mask_paths = zip(*valid_test_pairs)
        print(f"✓ Found {len(test_image_paths)} test samples")
        
        # Create test dataset and loader
        test_dataset = SimpleStarDistDataset(test_image_paths, test_mask_paths, 
                                            patch_size=config.patch_size, n_rays=config.n_rays)
        test_loader = DataLoader(test_dataset, batch_size=config.batch_size, 
                                shuffle=False, num_workers=2, pin_memory=True)
        
        print(f"✓ Test loader created: {len(test_loader)} batches\n")
        
        # Use optimal thresholds from validation
        opt_prob_thresh = best_f1['prob_thresh']
        opt_nms_thresh = best_f1['nms_thresh']
        
        print(f"Using optimal thresholds from validation:")
        print(f"  - Probability threshold: {opt_prob_thresh}")
        print(f"  - NMS threshold: {opt_nms_thresh}\n")
        
        # Evaluate on test set
        test_precisions = []
        test_recalls = []
        test_f1_scores = []
        test_accuracies = []
            ap_scores = []\n
        test_losses = []
        test_prob_preds = []
        test_boundary_preds = []
        test_images = []
        test_gt_masks = []
        
        model.eval()
        with torch.no_grad():
            pbar = tqdm(test_loader, desc="Evaluating test set")
            
            for images, prob_gt, dist_mask_gt, boundary_gt in pbar:
                images_gpu = images.to(config.device)
                prob_gt_gpu = prob_gt.to(config.device)
                dist_mask_gt_gpu = dist_mask_gt.to(config.device)
                boundary_gt_gpu = boundary_gt.to(config.device)
                
                # Forward pass
                outputs = inference_with_tta(model, images_gpu, config.n_rays)
                prob_pred = outputs['prob']
                dist_pred = outputs['dist']
                boundary_pred = outputs['boundary']
                
                # Compute loss
                loss, _ = total_loss_with_boundary(
                    prob_pred, dist_pred, boundary_pred,
                    prob_gt_gpu, dist_mask_gt_gpu, boundary_gt_gpu,
                    loss_weights=config.loss_weights
                )
                test_losses.append(loss.item())
                
                # Move to CPU for evaluation
                prob_pred_cpu = prob_pred.cpu().numpy()
                dist_pred_cpu = dist_pred.cpu().numpy()
                boundary_pred_cpu = boundary_pred.cpu().numpy()
                
                # Evaluate each sample
                for i in range(len(images)):
                    # Ground truth
                    gt_mask = (dist_mask_gt[i, -1].cpu().numpy() > 0).astype(np.int32)
                    gt_labeled, _ = scipy_label(gt_mask)
                    
                    # Prediction
                    pred_prob = prob_pred_cpu[i, 0]
                    pred_dist = dist_pred_cpu[i]
                    pred_mask = non_maximum_suppression(
                        pred_prob, pred_dist,
                        prob_thresh=opt_prob_thresh,
                        nms_thresh=opt_nms_thresh
                    )
                    
                    # Compute metrics
                    prec, rec, f1, acc = evaluate_detection(gt_labeled, pred_mask, iou_thresh=0.5)
                    ap_score = compute_dsb2018_ap(gt_labeled, pred_mask)\n
                    test_precisions.append(prec)
                    test_recalls.append(rec)
                    test_f1_scores.append(f1)
                    test_accuracies.append(acc)
                    ap_scores.append(ap_score)\n
                    
                    # Save first 10 samples for visualization
                    if len(test_images) < 10:
                        test_images.append(images[i, 0].cpu().numpy())
                        test_prob_preds.append(pred_prob)
                        test_boundary_preds.append(boundary_pred_cpu[i, 0])
                        test_gt_masks.append(gt_labeled)
                
                pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'f1': f'{np.mean(test_f1_scores):.3f}'
                })
        
        # Compute final test metrics
        test_loss = np.mean(test_losses)
        test_precision = np.mean(test_precisions)
        test_recall = np.mean(test_recalls)
        test_f1 = np.mean(test_f1_scores)
        test_accuracy = np.mean(test_accuracies)
        
        print("\n" + "="*60)
        print("TEST SET RESULTS")
        print("="*60)
        print(f"Test Loss: {test_loss:.4f}")
        print(f"Test Precision: {test_precision:.4f}")
        print(f"Test Recall: {test_recall:.4f}")
        print(f"Test F1 Score: {test_f1:.4f}")
        print(f"Test Accuracy: {test_accuracy:.4f}")
        print("="*60)
        
        # Save test results
        test_results = {
            'test_loss': test_loss,
            'test_precision': test_precision,
            'test_recall': test_recall,
            'test_f1': test_f1,
            'test_accuracy': test_accuracy,
            'num_samples': len(test_dataset),
            'prob_threshold': opt_prob_thresh,
            'nms_threshold': opt_nms_thresh
        }
        
        import json
        with open(os.path.join(config.save_dir, 'test_results.json'), 'w') as f:
            json.dump(test_results, f, indent=2)
        
        print(f"\n✓ Test results saved to {config.save_dir}/test_results.json")
        
    except Exception as e:
        print(f"❌ Error during test evaluation: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"⚠️ Test data not found at {TEST_DATA_ROOT}")
    print("Skipping test evaluation")

## 15. Visualize Test Predictions

In [ ]:
# ============================================================================
# VISUALIZE TEST PREDICTIONS
# ============================================================================

if 'test_images' in locals() and len(test_images) > 0:
    print("\nVisualizing test predictions...")
    
    num_samples = min(5, len(test_images))
    fig, axes = plt.subplots(4, num_samples, figsize=(4*num_samples, 16))
    
    if num_samples == 1:
        axes = axes.reshape(-1, 1)
    
    for i in range(num_samples):
        # Original image
        axes[0, i].imshow(test_images[i], cmap='gray')
        axes[0, i].set_title(f'Test Sample {i+1}\nOriginal Image', fontweight='bold')
        axes[0, i].axis('off')
        
        # Ground truth mask
        axes[1, i].imshow(test_gt_masks[i], cmap='nipy_spectral')
        axes[1, i].set_title('Ground Truth Mask', fontweight='bold')
        axes[1, i].axis('off')
        
        # Probability prediction
        im2 = axes[2, i].imshow(test_prob_preds[i], cmap='hot', vmin=0, vmax=1)
        axes[2, i].set_title(f'Probability Prediction\n(thresh={opt_prob_thresh})', fontweight='bold')
        axes[2, i].axis('off')
        plt.colorbar(im2, ax=axes[2, i], fraction=0.046)
        
        # Boundary prediction
        im3 = axes[3, i].imshow(test_boundary_preds[i], cmap='hot', vmin=0, vmax=1)
        axes[3, i].set_title('Boundary Prediction', fontweight='bold')
        axes[3, i].axis('off')
        plt.colorbar(im3, ax=axes[3, i], fraction=0.046)
    
    plt.tight_layout()
    plt.savefig(os.path.join(config.save_dir, 'test_predictions.png'), 
                dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Test predictions saved to {config.save_dir}/test_predictions.png")
else:
    print("No test predictions to visualize")

## 16. Comparison: Validation vs Test Performance

In [ ]:
# ============================================================================
# VAL vs TEST COMPARISON
# ============================================================================

if 'test_precision' in locals():
    print("\nCreating Val vs Test comparison...")
    
    # Create comparison plot
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    
    metrics = ['Precision', 'Recall', 'F1 Score']
    val_scores = [best_f1['precision'], best_f1['recall'], best_f1['f1']]
    test_scores = [test_precision, test_recall, test_f1]
    
    x = np.arange(len(metrics))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, val_scores, width, label='Validation', 
                   color='#3498db', alpha=0.8, edgecolor='black', linewidth=1.5)
    bars2 = ax.bar(x + width/2, test_scores, width, label='Test', 
                   color='#e74c3c', alpha=0.8, edgecolor='black', linewidth=1.5)
    
    ax.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax.set_title('Validation vs Test Performance', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics, fontsize=11)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim([0, 1.0])
    
    # Add value labels on bars
    def autolabel(bars):
        for bar in bars:
            height = bar.get_height()
            ax.annotate(f'{height:.3f}',
                       xy=(bar.get_x() + bar.get_width() / 2, height),
                       xytext=(0, 3),
                       textcoords="offset points",
                       ha='center', va='bottom',
                       fontsize=9, fontweight='bold')
    
    autolabel(bars1)
    autolabel(bars2)
    
    plt.tight_layout()
    plt.savefig(os.path.join(config.save_dir, 'val_vs_test.png'), 
                dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Comparison saved to {config.save_dir}/val_vs_test.png")
    
    # Print comparison table
    print("\n" + "="*60)
    print("VALIDATION vs TEST COMPARISON")
    print("="*60)
    print(f"{'Metric':<15} {'Validation':>12} {'Test':>12} {'Diff':>12}")
    print("-" * 60)
    for metric, val, test in zip(metrics, val_scores, test_scores):
        diff = test - val
        diff_str = f"+{diff:.3f}" if diff >= 0 else f"{diff:.3f}"
        print(f"{metric:<15} {val:>12.4f} {test:>12.4f} {diff_str:>12}")
    print("="*60)
else:
    print("No test results for comparison")

## 17. Final Summary & Download Results

In [ ]:
# ============================================================================
# FINAL COMPREHENSIVE SUMMARY
# ============================================================================

# Update summary with test results
test_summary = f"""
{'='*70}
STARDIST CELL SEGMENTATION - COMPREHENSIVE TRAINING SUMMARY
{'='*70}

MODEL CONFIGURATION:
  - Architecture: Attention-Enhanced U-Net
  - Depth: {config.unet_n_depth}
  - Base filters: {config.unet_n_filter_base}
  - Conv after UNet: {config.net_conv_after_unet}
  - Total parameters: {total_params:,}
  - Attention: {config.use_attention}
  - Attention Gates: {config.use_attention_gate}
  - Boundary Head: {config.use_boundary_head}

TRAINING CONFIGURATION:
  - Epochs trained: {len(history['train_loss'])}
  - Batch size: {config.batch_size}
  - Learning rate: {config.learning_rate}
  - Loss weights: {config.loss_weights}
  - Train samples: {len(train_loader.dataset)}
  - Val samples: {len(val_loader.dataset)}
"""

if 'test_precision' in locals():
    test_summary += f"""
  - Test samples: {len(test_dataset)}

VALIDATION RESULTS:
  - Best validation loss: {best_val_loss:.4f}
  - Best F1 score: {best_f1['f1']:.3f}
  - Best precision: {best_f1['precision']:.3f}
  - Best recall: {best_f1['recall']:.3f}
  - Optimal prob threshold: {best_f1['prob_thresh']}
  - Optimal NMS threshold: {best_f1['nms_thresh']}

TEST RESULTS:
  - Test loss: {test_loss:.4f}
  - Test F1 score: {test_f1:.3f}
  - Test precision: {test_precision:.3f}
  - Test recall: {test_recall:.3f}

GENERALIZATION (Val → Test):
  - Precision: {best_f1['precision']:.3f} → {test_precision:.3f} ({test_precision - best_f1['precision']:+.3f})
  - Recall: {best_f1['recall']:.3f} → {test_recall:.3f} ({test_recall - best_f1['recall']:+.3f})
  - F1 Score: {best_f1['f1']:.3f} → {test_f1:.3f} ({test_f1 - best_f1['f1']:+.3f})
"""
else:
    test_summary += f"""

VALIDATION RESULTS:
  - Best validation loss: {best_val_loss:.4f}
  - Best F1 score: {best_f1['f1']:.3f}
  - Best precision: {best_f1['precision']:.3f}
  - Best recall: {best_f1['recall']:.3f}
  - Optimal prob threshold: {best_f1['prob_thresh']}
  - Optimal NMS threshold: {best_f1['nms_thresh']}

TEST RESULTS:
  - Test data not available
"""

test_summary += f"""

OUTPUT FILES:
  - Best model: {config.save_dir}/best_model.pth
  - Training history: {config.save_dir}/training_history.png
  - Threshold evaluation: {config.save_dir}/threshold_evaluation.csv
  - Threshold analysis: {config.save_dir}/threshold_analysis.png
  - Sample predictions: {config.save_dir}/predictions_sample.png
"""

if 'test_precision' in locals():
    test_summary += f"""  - Test results: {config.save_dir}/test_results.json
  - Test predictions: {config.save_dir}/test_predictions.png
  - Val vs Test comparison: {config.save_dir}/val_vs_test.png
"""

test_summary += f"""

KAGGLE DOWNLOAD:
  All files are in: {config.save_dir}/
  Download from Kaggle Output panel or use:
  !zip -r results.zip {config.save_dir}

{'='*70}
"""

print(test_summary)

# Save comprehensive summary
with open(os.path.join(config.save_dir, 'final_summary.txt'), 'w') as f:
    f.write(test_summary)

print(f"\n✓ Final summary saved to {config.save_dir}/final_summary.txt")
print(f"\n{'='*70}")
print(f"✅ ALL DONE! Training and evaluation completed successfully.")
print(f"{'='*70}")
print(f"\n📦 To download all results from Kaggle:")
print(f"   Run: !zip -r results.zip {config.save_dir}")
print(f"   Then download 'results.zip' from Output panel")

## 18. Optional: Create Results Archive

In [ ]:
# Create zip archive for easy download
import zipfile

zip_path = os.path.join(WORK_DIR, "stardist_results.zip")

print("Creating results archive...")
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(config.save_dir):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, WORK_DIR)
            zipf.write(file_path, arcname)
            print(f"  Added: {arcname}")

zip_size_mb = os.path.getsize(zip_path) / (1024**2)
print(f"\n✓ Archive created: {zip_path}")
print(f"✓ Size: {zip_size_mb:.2f} MB")
print(f"\n📥 Download this file from Kaggle Output panel")